# 🤖 BERTopic Training on Kaggle GPU

**Member 3 - Task 3.1: Train BERTopic với PhoBERT**

---

## 📋 Setup Instructions

### Trước khi chạy:

1. **Bật GPU trên Kaggle:**
   - Settings (bên phải) → Accelerator → GPU T4 x2

2. **Upload data:**
   - Add Data → Upload → `result.csv` từ `data/preprocessed/`
   - Hoặc: Upload lên Kaggle Dataset trước

3. **Chạy từng cell theo thứ tự!**

---

## ⏱️ Thời gian ước tính:
- Install packages: 3-5 phút
- Train BERTopic: 20-40 phút (tùy số documents)
- **TOTAL: ~30-45 phút**

---
## 📦 CELL 1: Install Dependencies

In [1]:
%%time
# Install tất cả packages cần thiết

print("📦 Installing dependencies...\n")

!pip install -q bertopic==0.17.4
!pip install -q sentence-transformers==5.3.0
!pip install -q umap-learn==0.5.11
!pip install -q hdbscan==0.8.42
!pip install -q gensim==4.4.0

print("\n✅ Installation complete!")

# Verify GPU
import torch
print(f"\n🔧 GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

📦 Installing dependencies...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 9.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 42.2 MB/s eta 0:00:0000:0100:01

✅ Installation complete!

🔧 GPU available: True
   GPU name: Tesla T4
   GPU memory: 15.6 GB
CPU times: user 2.25 s, sys: 854 ms, total: 3.11 s
Wall time: 40.7 s


---
## 📊 CELL 2: Load Data

In [2]:
import pandas as pd
import numpy as np

print("📊 Loading data...\n")

# OPTION 1: Nếu upload trực tiếp lên notebook
# df = pd.read_csv('/kaggle/input/result.csv')

# OPTION 2: Nếu upload lên Kaggle Dataset
# df = pd.read_csv('/kaggle/input/your-dataset-name/result.csv')

# OPTION 3: Upload thủ công qua UI (thay đổi path)
df = pd.read_csv('/kaggle/input/datasets/anhqunhong/vietnamese-tech-comments/result.csv')  # ← SỬA PATH NÀY!

# Load documents
documents = df['clean_text'].dropna().tolist()

print(f"✅ Loaded {len(documents):,} documents")
print(f"\nSample documents:")
for i, doc in enumerate(documents[:3], 1):
    print(f"{i}. {doc[:80]}...")

📊 Loading data...

✅ Loaded 150 documents

Sample documents:
1. bài đánh_giá tâm quá thanks_thím chia_sẻ anh_em mở_mang tầm_mắt...
2. cấu_hình ngon đấy người tạo bài cứ thế mà làm luôn thôi không phải nghĩ...
3. mình nghĩ nên đợi thêm một thời_gian để xem các thử_thách tình_cảm nhiệt_độ độc_...


---
## 🤖 CELL 3: Define BERTopic Model Class

In [3]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from typing import List, Tuple
import pickle
import os

class VietnameseBERTopicModel:
    """
    BERTopic wrapper cho tiếng Việt với PhoBERT
    """
    
    def __init__(
        self,
        embedding_model: str = "vinai/phobert-base",
        n_neighbors: int = 15,
        n_components: int = 5,
        min_cluster_size: int = 15,
        min_samples: int = 10,
        top_n_words: int = 10,
        verbose: bool = True,
        use_gpu: bool = True
    ):
        self.verbose = verbose
        self.embedding_model_name = embedding_model
        
        # GPU detection
        if use_gpu and torch.cuda.is_available():
            self.device = 'cuda'
            if self.verbose:
                print(f"🚀 Using GPU: {torch.cuda.get_device_name(0)}")
        else:
            self.device = 'cpu'
            if self.verbose:
                print("⚠️  Using CPU")
        
        # Save params
        self.n_neighbors = n_neighbors
        self.n_components = n_components
        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        self.top_n_words = top_n_words
        
        # Initialize PhoBERT
        if self.verbose:
            print(f"\n[1/4] Loading {embedding_model}...")
        self.embedding_model = SentenceTransformer(embedding_model, device=self.device)
        
        # Initialize UMAP
        if self.verbose:
            print(f"[2/4] Configuring UMAP...")
        self.umap_model = UMAP(
            n_neighbors=n_neighbors,
            n_components=n_components,
            min_dist=0.0,
            metric='cosine',
            random_state=42
        )
        
        # Initialize HDBSCAN
        if self.verbose:
            print(f"[3/4] Configuring HDBSCAN...")
        self.hdbscan_model = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True
        )
        
        # Initialize BERTopic
        if self.verbose:
            print(f"[4/4] Building BERTopic pipeline...")
        self.topic_model = BERTopic(
            embedding_model=self.embedding_model,
            umap_model=self.umap_model,
            hdbscan_model=self.hdbscan_model,
            top_n_words=top_n_words,
            verbose=verbose,
            calculate_probabilities=True
        )
        
        if self.verbose:
            print("✅ Model initialized!\n")
        
        self.topics_ = None
        self.probs_ = None
    
    def fit(self, documents: List[str]) -> Tuple:
        """Train BERTopic"""
        if self.verbose:
            print(f"\n{'='*60}")
            print("TRAINING BERTOPIC")
            print(f"{'='*60}")
            print(f"Documents: {len(documents):,}")
        
        topics, probs = self.topic_model.fit_transform(documents)
        
        self.topics_ = topics
        self.probs_ = probs
        
        if self.verbose:
            n_topics = len(set(topics)) - (1 if -1 in topics else 0)
            n_outliers = sum(t == -1 for t in topics)
            print(f"\n{'='*60}")
            print("✅ TRAINING COMPLETED")
            print(f"{'='*60}")
            print(f"Topics found: {n_topics}")
            print(f"Outliers: {n_outliers} ({n_outliers/len(topics)*100:.1f}%)")
            print(f"{'='*60}\n")
        
        return topics, probs
    
    def get_topic_info(self) -> pd.DataFrame:
        """Get topic information"""
        return self.topic_model.get_topic_info()
    
    def get_topics(self, topic_id=None):
        """Get top words for topic(s)"""
        if topic_id is not None:
            topic = self.topic_model.get_topic(topic_id)
            return {topic_id: topic} if topic else {}
        else:
            all_topics = {}
            for tid in set(self.topics_):
                if tid != -1:
                    topic = self.topic_model.get_topic(tid)
                    if topic:
                        all_topics[tid] = topic
            return all_topics
    
    def calculate_coherence(self, documents: List[str], coherence_type: str = 'c_v') -> float:
        """Calculate coherence score"""
        if self.verbose:
            print(f"\nCalculating coherence ({coherence_type})...")
        
        topics_dict = self.get_topics()
        if not topics_dict:
            return 0.0
        
        topics_words = []
        for topic_id in sorted(topics_dict.keys()):
            words = [word for word, score in topics_dict[topic_id]]
            topics_words.append(words)
        
        texts = [doc.split() for doc in documents]
        dictionary = Dictionary(texts)
        
        coherence_model = CoherenceModel(
            topics=topics_words,
            texts=texts,
            dictionary=dictionary,
            coherence=coherence_type,
            processes=1  # Important for Kaggle!
        )
        
        coherence_score = coherence_model.get_coherence()
        
        if self.verbose:
            print(f"✅ Coherence {coherence_type.upper()}: {coherence_score:.4f}")
        
        return coherence_score
    
    def save(self, path: str):
        """Save model"""
        os.makedirs(path, exist_ok=True)
        
        # Save BERTopic
        bertopic_path = os.path.join(path, "bertopic_model")
        self.topic_model.save(bertopic_path, serialization="pickle")
        
        # Save config
        config = {
            'embedding_model': self.embedding_model_name,
            'n_neighbors': self.n_neighbors,
            'n_components': self.n_components,
            'min_cluster_size': self.min_cluster_size,
            'min_samples': self.min_samples,
            'top_n_words': self.top_n_words
        }
        
        with open(os.path.join(path, "config.pkl"), 'wb') as f:
            pickle.dump(config, f)
        
        # Save topics
        if self.topics_ is not None:
            with open(os.path.join(path, "topics.pkl"), 'wb') as f:
                pickle.dump({'topics': self.topics_, 'probs': self.probs_}, f)
        
        if self.verbose:
            print(f"✅ Model saved to {path}")

print("✅ BERTopic class defined!")

2026-04-04 15:31:00.794780: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775316660.994363      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775316661.054684      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775316661.536583      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775316661.536620      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775316661.536623      55 computation_placer.cc:177] computation placer alr

✅ BERTopic class defined!


---
## 🚀 CELL 4: Initialize Model

In [4]:
%%time

# Khởi tạo model với hyperparameters
model = VietnameseBERTopicModel(
    embedding_model="vinai/phobert-base",
    n_neighbors=15,
    n_components=5,
    min_cluster_size=15,
    min_samples=10,
    top_n_words=10,
    verbose=True,
    use_gpu=True  # Sử dụng GPU
)

🚀 Using GPU: Tesla T4

[1/4] Loading vinai/phobert-base...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[2/4] Configuring UMAP...
[3/4] Configuring HDBSCAN...
[4/4] Building BERTopic pipeline...
✅ Model initialized!

CPU times: user 7.08 s, sys: 7.36 s, total: 14.4 s
Wall time: 8.32 s


---
## 🏋️ CELL 5: Train Model (Lâu nhất - 20-40 phút)

In [5]:
%%time

# Train BERTopic
print("🏋️ Starting training...\n")

import time
start_time = time.time()

topics, probs = model.fit(documents)

train_time = time.time() - start_time

print(f"\n⏱️ Training time: {train_time/60:.1f} minutes")

2026-04-04 15:33:03,329 - BERTopic - Embedding - Transforming documents to embeddings.


🏋️ Starting training...


TRAINING BERTOPIC
Documents: 150


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

2026-04-04 15:33:04,035 - BERTopic - Embedding - Completed ✓
2026-04-04 15:33:04,036 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-04 15:33:10,870 - BERTopic - Dimensionality - Completed ✓
2026-04-04 15:33:10,871 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-04 15:33:10,887 - BERTopic - Cluster - Completed ✓
2026-04-04 15:33:10,895 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-04 15:33:10,917 - BERTopic - Representation - Completed ✓



✅ TRAINING COMPLETED
Topics found: 4
Outliers: 27 (18.0%)


⏱️ Training time: 0.1 minutes
CPU times: user 6.92 s, sys: 362 ms, total: 7.29 s
Wall time: 7.61 s


---
## 📊 CELL 6: Analyze Topics

In [6]:
# Get topic info
topic_info = model.get_topic_info()

print("📊 TOPIC SUMMARY:\n")
print(topic_info)

# Display top 5 topics
print("\n" + "="*60)
print("🏆 TOP 5 TOPICS")
print("="*60 + "\n")

topic_info_sorted = topic_info[topic_info['Topic'] != -1].sort_values('Count', ascending=False)

for i in range(min(5, len(topic_info_sorted))):
    row = topic_info_sorted.iloc[i]
    topic_id = row['Topic']
    topic_name = row['Name']
    count = row['Count']
    
    print(f"Topic {topic_id}: {topic_name}")
    print(f"  Documents: {count}")
    
    # Get top words
    top_words = model.get_topics(topic_id)
    if topic_id in top_words:
        words = [f"{word}({score:.3f})" for word, score in top_words[topic_id][:5]]
        print(f"  Top words: {', '.join(words)}")
    print()

📊 TOPIC SUMMARY:

   Topic  Count                                Name  \
0     -1     27                -1_mua_triệu_thím_30   
1      0     54             0_giá_người_rẻ_cấu_hình   
2      1     34                    1_bài_cái_quá_em   
3      2     20           2_mình_cuối_cùng_một_hẵng   
4      3     15  3_thay_đổi_ram_thông_số_dung_lượng   

                                      Representation  \
0  [mua, triệu, thím, 30, gần, bỏ, iphone, hối_hậ...   
1  [giá, người, rẻ, cấu_hình, mà, tiền, con, thì,...   
2  [bài, cái, quá, em, viết, ghim, cũ_rích, ai, l...   
3  [mình, cuối_cùng, một, hẵng, thử_thách, thời_g...   
4  [thay_đổi, ram, thông_số, dung_lượng, công_bố,...   

                                 Representative_Docs  
0  [bỏ gần 30 triệu mua iphone mới thấy hối_hận q...  
1  [cấu_hình ngon đấy người tạo bài cứ thế mà làm...  
2  [lại bài cũ_rích mua đi rồi khóc nóng_như cái ...  
3  [mình nghĩ nên đợi thêm một thời_gian để xem c...  
4  [theo thông_số công_bố thì chỉ thay_

---
## 📈 CELL 7: Calculate Coherence (Optional - có thể bỏ qua nếu lâu)

In [7]:
%%time

# Tính coherence score
# NOTE: Có thể mất 10-20 phút! Bỏ qua cell này nếu muốn nhanh.

print("📈 Calculating coherence score...\n")
coherence = model.calculate_coherence(documents, coherence_type='c_v')

print(f"\n✅ Coherence C_V: {coherence:.4f}")

📈 Calculating coherence score...


Calculating coherence (c_v)...
✅ Coherence C_V: 0.5736

✅ Coherence C_V: 0.5736
CPU times: user 54.2 ms, sys: 2.02 ms, total: 56.2 ms
Wall time: 56.7 ms


---
## 💾 CELL 8: Save Results

In [8]:
# Tạo output folder
os.makedirs("/kaggle/working/output", exist_ok=True)

# Save topic info to CSV
topic_info.to_csv("/kaggle/working/output/bertopic_topics.csv", index=False)
print("✅ Saved: bertopic_topics.csv")

# Save model
model.save("/kaggle/working/output/bertopic_model")
print("✅ Saved: bertopic_model/")

# Save summary
summary = {
    'n_documents': len(documents),
    'n_topics': len(set(topics)) - (1 if -1 in topics else 0),
    'n_outliers': sum(t == -1 for t in topics),
    'outlier_ratio': sum(t == -1 for t in topics) / len(topics),
    'training_time_minutes': train_time / 60,
    'coherence_cv': coherence if 'coherence' in locals() else None,
    'hyperparameters': {
        'n_neighbors': 15,
        'n_components': 5,
        'min_cluster_size': 15,
        'min_samples': 10
    }
}

with open("/kaggle/working/output/summary.pkl", 'wb') as f:
    pickle.dump(summary, f)
print("✅ Saved: summary.pkl")

print("\n" + "="*60)
print("💾 ALL RESULTS SAVED!")
print("="*60)
print("\n📂 Files in /kaggle/working/output/:")
!ls -lh /kaggle/working/output/

2026-04-04 15:33:11,039 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✅ Saved: bertopic_topics.csv
✅ Model saved to /kaggle/working/output/bertopic_model
✅ Saved: bertopic_model/
✅ Saved: summary.pkl

💾 ALL RESULTS SAVED!

📂 Files in /kaggle/working/output/:
total 12K
drwxr-xr-x 2 root root 4.0K Apr  4 15:33 bertopic_model
-rw-r--r-- 1 root root 2.4K Apr  4 15:33 bertopic_topics.csv
-rw-r--r-- 1 root root  330 Apr  4 15:33 summary.pkl


---
## 📋 CELL 9: Final Summary

In [9]:
print("="*70)
print("🎉 TRAINING SUMMARY")
print("="*70)
print(f"Documents: {len(documents):,}")
print(f"Topics found: {len(set(topics)) - (1 if -1 in topics else 0)}")
print(f"Outliers: {sum(t == -1 for t in topics)} ({sum(t == -1 for t in topics)/len(topics)*100:.1f}%)")
if 'coherence' in locals():
    print(f"Coherence C_V: {coherence:.4f}")
print(f"Training time: {train_time/60:.1f} minutes")
print(f"Device: {model.device}")
print("="*70)

print("\n📥 DOWNLOAD INSTRUCTIONS:")
print("1. Click Output tab (bên phải)")
print("2. Download files:")
print("   - bertopic_topics.csv")
print("   - bertopic_model/ (folder)")
print("   - summary.pkl")
print("\n3. Copy về project local:")
print("   output/bertopic_model_kaggle/")

print("\n✅ DONE! Chúc mừng bạn đã train xong BERTopic! 🎉")

🎉 TRAINING SUMMARY
Documents: 150
Topics found: 4
Outliers: 27 (18.0%)
Coherence C_V: 0.5736
Training time: 0.1 minutes
Device: cuda

📥 DOWNLOAD INSTRUCTIONS:
1. Click Output tab (bên phải)
2. Download files:
   - bertopic_topics.csv
   - bertopic_model/ (folder)
   - summary.pkl

3. Copy về project local:
   output/bertopic_model_kaggle/

✅ DONE! Chúc mừng bạn đã train xong BERTopic! 🎉


---
## 📝 NOTES

### Nếu gặp lỗi:

**Out of Memory:**
- Giảm `min_cluster_size` xuống 10
- Hoặc sample dataset nhỏ hơn

**Quá lâu:**
- Bỏ qua Cell 7 (coherence calculation)
- Hoặc sample 10K documents thử trước

**GPU không hoạt động:**
- Check Settings → Accelerator → GPU T4 x2
- Restart notebook

---

### Tips:
- ✅ Save notebook thường xuyên!
- ✅ Kaggle có 30 giờ GPU miễn phí/tuần
- ✅ Download results ngay sau khi xong
- ✅ Có thể train nhiều configs khác nhau

---

**Created:** 2026-04-04  
**Author:** Member 3 (ML Engineer)  
**Task:** 3.1 - BERTopic Implementation